# Function calling nativo 

Anteriormente construimos un agente que pedía herramientas con un **JSON escrito a mano**. Hoy aprendemos la forma **estándar y más fiable** de hacerlo: el **function calling nativo**.

La idea es la misma de siempre: **el modelo decide, nosotros ejecutamos**. La diferencia es que ahora le pasamos las herramientas en un parámetro especial (`tools`) y el modelo nos contesta de forma **estructurada** diciéndonos qué función llamar.


> **Importante:** el function calling nativo necesita un modelo compatible. **Groq** (con `llama-3.3-70b-versatile`) va muy bien. En OpenRouter, elige un modelo que soporte *tools*.


## 0. Configuración

Instala la librería y crea el cliente (igual que en el Día 1).

In [1]:
from openai import OpenAI
from getpass import getpass
import json

API_KEY = getpass("Pega aquí tu clave API: ")
BASE_URL = "https://api.groq.com/openai/v1"
MODELO   = "llama-3.3-70b-versatile"

cliente = OpenAI(api_key=API_KEY, base_url=BASE_URL)

## Recordatorio: ¿qué problema resolvemos?

Un LLM solo genera texto: no sabe la hora real ni calcula con fiabilidad. Le damos **herramientas** (funciones de Python) y dejamos que decida cuándo usarlas.

Empecemos con dos herramientas sencillas.

In [2]:
import datetime, ast, operator

def hora_actual():
    
    return datetime.datetime.now().strftime("%d/%m/%Y %H:%M:%S")


_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod, ast.USub: operator.neg}
def _ev(n):
    if isinstance(n, ast.Constant): return n.value
    if isinstance(n, ast.BinOp):   return _OPS[type(n.op)](_ev(n.left), _ev(n.right))
    if isinstance(n, ast.UnaryOp): return _OPS[type(n.op)](_ev(n.operand))
    raise ValueError("no permitido")
def calculadora(operacion):
    
    try:    return str(_ev(ast.parse(operacion, mode="eval").body))
    except Exception: return "Error: operación no válida"

print(hora_actual())
print(calculadora("48273 * 9912"))

29/06/2026 09:26:22
478481976


## La "ficha" de cada herramienta (el *schema*)

Para que el modelo sepa que existen y cómo usarlas, se las describimos con un **esquema** en formato de diccionario. Cada herramienta tiene:

- **name:** el nombre exacto de la función.
- **description:** qué hace y *cuándo* usarla (¡muy importante para que acierte!).
- **parameters:** qué datos necesita y de qué tipo.

> Los **nombres de los parámetros** del esquema deben coincidir con los de la función de Python.

In [4]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "hora_actual",
            "description": "Devuelve la fecha y la hora actuales. Úsala cuando pregunten qué hora o qué día es.",
            "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculadora",
            "description": "Hace cálculos matemáticos exactos. Úsala para sumar, restar, multiplicar, etc.",
            "parameters": {
                "type": "object",
                "properties": {
                    "operacion": {"type": "string", "description": "La operación a calcular, p. ej. '23*5+1'"}
                },
                "required": ["operacion"],
            },
        },
    },
]

funciones = {"hora_actual": hora_actual, "calculadora": calculadora}


## Primera llamada con `tools`: mira cómo el modelo "pide"

Hacemos una llamada **pasando las herramientas**. Fíjate: el modelo no responde con texto, sino con una **petición de herramienta** (`tool_calls`).

In [5]:
mensajes = [{"role": "user", "content": "¿Cuánto es 48273 por 9912?"}]

respuesta = cliente.chat.completions.create(
    model=MODELO, messages=mensajes, tools=tools, tool_choice="auto",
)
msg = respuesta.choices[0].message

print("¿Texto directo?:", msg.content)
print("¿Pide herramientas?:")
for tc in (msg.tool_calls or []):
    print("  - función:", tc.function.name, "| argumentos:", tc.function.arguments)

¿Texto directo?: None
¿Pide herramientas?:
  - función: calculadora | argumentos: {"operacion":"48273*9912"}


¿Lo ves? En lugar de inventarse el número, el modelo ha pedido usar `calculadora` con la operación correcta. Los `arguments` vienen como **texto JSON**, así que los convertiremos con `json.loads`.

## 4. Ejecutamos la herramienta y se la devolvemos

Ahora cerramos el círculo **a mano** una vez (para entenderlo) y luego lo automatizamos:
1. Ejecutamos la función que pidió.
2. Le devolvemos el resultado con un mensaje de **rol `tool`**.
3. El modelo redacta la respuesta final.

### 1) Guardamos lo que pidió el modelo

In [6]:
import json

mensajes.append({
    "role": "assistant", "content": msg.content,
    "tool_calls": [{"id": tc.id, "type": "function",
                    "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                   for tc in msg.tool_calls],
})

In [7]:
mensajes

[{'role': 'user', 'content': '¿Cuánto es 48273 por 9912?'},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [{'id': '5r70xdfzy',
    'type': 'function',
    'function': {'name': 'calculadora',
     'arguments': '{"operacion":"48273*9912"}'}}]}]

### 2) Ejecutamos cada herramienta y devolvemos el resultado

In [10]:
for tc in msg.tool_calls:
    args = json.loads(tc.function.arguments)
    resultado = funciones[tc.function.name](**args)
    mensajes.append({"role": "tool", "tool_call_id": tc.id, "content": str(resultado)})

In [11]:
mensajes

[{'role': 'user', 'content': '¿Cuánto es 48273 por 9912?'},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [{'id': '5r70xdfzy',
    'type': 'function',
    'function': {'name': 'calculadora',
     'arguments': '{"operacion":"48273*9912"}'}}]},
 {'role': 'tool', 'tool_call_id': '5r70xdfzy', 'content': '478481976'}]

### 3) Segunda llamada: ahora el modelo ya tiene el resultado

In [12]:

respuesta2 = cliente.chat.completions.create(model=MODELO, messages=mensajes, tools=tools)
print(respuesta2.choices[0].message.content)

El resultado de 48273 por 9912 es 478481976.


## 5. El bucle completo (automatizado)

Hacer esto a mano cada vez es tedioso. Vamos a meterlo en una función reutilizable, `ejecutar_agente`, que repite el proceso hasta que el modelo dé su respuesta final. **Guárdala bien: la usaremos en todos los notebooks de hoy.**

In [13]:
def ejecutar_agente(mensajes, tools, funciones, max_pasos=5, verbose=True):
    
    for paso in range(1, max_pasos + 1):
        respuesta = cliente.chat.completions.create(
            model=MODELO,
            messages=mensajes,
            tools=tools,            
            tool_choice="auto",     
            temperature=0,
        )
        msg = respuesta.choices[0].message

        if not msg.tool_calls:
            return msg.content

        mensajes.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ],
        })

        for tc in msg.tool_calls:
            nombre = tc.function.name
            argumentos_crudos = json.loads(tc.function.arguments or "{}")
            
            if not isinstance(argumentos_crudos, dict):
                argumentos = {}
            else:
                argumentos = argumentos_crudos

            funcion = funciones.get(nombre)
            if funcion is None:
                resultado = f"Error: la herramienta '{nombre}' no existe."
            else:
                try:
                    resultado = funcion(**argumentos)
                except Exception as e:
                    resultado = f"Error al ejecutar {nombre}: {e}"
            if verbose:
                print(f"[Paso {paso}] {nombre}({argumentos}) → {str(resultado)[:120]}")
            mensajes.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(resultado),
            })

    return "He alcanzado el límite de pasos"

Probémosla. Le hacemos una pregunta que necesita **dos** herramientas seguidas (la hora y un cálculo).

In [21]:
mensajes = [
    {"role": "system", "content": "Eres un asistente que usa herramientas cuando hace falta. No intentes utilizar funciones de manera asincrona"},
    {"role": "user", "content": "¿Qué hora es? Y de paso, dime cuánto es 1234 * 5678."},
]
respuesta_final = ejecutar_agente(mensajes, tools, funciones)
print("Respuesta final:", respuesta_final)

[Paso 1] hora_actual({}) → 29/06/2026 09:44:34
[Paso 1] calculadora({'operacion': '1234 * 5678'}) → 7006652
Respuesta final: La hora actual es 09:44:34 y el resultado de la operación 1234 * 5678 es 7006652.


## Añadir más herramientas

Para ampliar al agente, solo hay que: 
- 1) escribir la función. 
- 2) añadir su ficha a `tools`.
- 3) registrarla en `funciones`. 


Vamos a darle un mini-catálogo de tienda.

In [ ]:
CATALOGO = {"portátil": 799, "ratón": 19, "monitor": 229}

def precio_producto(nombre):
    for clave, precio in CATALOGO.items():
        if clave in nombre.lower():
            return f"{clave}: {precio} €"
    return "Producto no encontrado."

tools.append({
    "type": "function",
    "function": {
        "name": "precio_producto",
        "description": "Consulta el precio de un producto de la tienda (portátil, ratón, monitor).",
        "parameters": {
            "type": "object",
            "properties": {"nombre": {"type": "string", "description": "Nombre del producto"}},
            "required": ["nombre"],
        },
    },
})

funciones["precio_producto"] = precio_producto

mensajes = [
    {"role": "user", "content": "¿Cuánto me costarían 3 ratones de la tienda?"}]


print(ejecutar_agente(mensajes, tools, funciones))

[Paso 1] precio_producto({'nombre': 'ratón'}) → ratón: 19 €
[Paso 1] calculadora({'operacion': '3 * resultado'}) → Error: operación no válida
[Paso 2] calculadora({'operacion': '3 * 19'}) → 57
Te costarían 57 euros.


In [27]:
mensajes = [
    {
        "role": "system", 
        "content": "Eres un asistente de tienda. MUY IMPORTANTE: No intentes anidar funciones ni hacer múltiples llamadas a herramientas en una sola acción. Trabaja paso a paso. Si necesitas calcular un total, PRIMERO usa la herramienta de precios y espera el resultado. LUEGO, en un paso independiente, usa la calculadora con el número exacto que obtuviste."
    },
    {
        "role": "user", 
        "content": "¿Cuánto me costarían 3 ratones de la tienda?"
    }
]

In [28]:
print(ejecutar_agente(mensajes, tools, funciones))

[Paso 1] precio_producto({'nombre': 'ratón'}) → ratón: 19 €
[Paso 2] calculadora({'operacion': '3*19'}) → 57
Los 3 ratones te costarían 57 €.


## El parámetro `tool_choice`

Controla cómo de "libre" es el modelo para usar herramientas:
- `"auto"` (por defecto): el modelo decide. Es lo normal.
- `"none"`: prohíbe usar herramientas (solo texto).
- `"required"`: le obliga a usar **alguna** herramienta.

Para la mayoría de casos, deja `"auto"`.

_________________________________________